<a href="https://colab.research.google.com/github/Shreenitya/nitya/blob/main/Animation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import geemap.geemap as map
import ee
ee.Authenticate()
ee.Initialize(project ='shreenitya0428')
m = map.Map()

In [2]:
India_boundary = ee.FeatureCollection('projects/shreenitya0428/assets/national_boundary')
m.addLayer(India_boundary,{},'Indian_Boundary')

In [3]:
region = ee.Geometry.Polygon(
    [[[67.060547, 4.740675],
     [67.060547, 38.065392],
     [98.964844, 38.065392],
     [98.964844, 4.740675],
     [67.060547, 4.740675]]],
    None,
    False)

In [5]:
ndvi = ee.ImageCollection("MODIS/061/MOD13A2").select('NDVI')

In [6]:
ndvi_India = ndvi.map(lambda img: img.set('doy',ee.Date(img.get('system:time_start')).getRelative('day', 'year')));
distinctDOY = ndvi_India.filterDate('2023-02-18', '2024-02-18');

In [7]:
filter = ee.Filter.equals(leftField ='doy',rightField = 'doy')
join = ee.Join.saveAll('doy_matches')
joincol = ee.ImageCollection(join.apply(distinctDOY,ndvi_India,filter))

In [8]:
composite = joincol.map(lambda img: ee.ImageCollection.fromImages(img.get('doy_matches')).reduce(ee.Reducer.mean()).set('doy',ee.Number(img.get('doy'))))

In [9]:
Vis_para = {
  'min': 2000,
  'max': 10000,
  'palette': [
    'ffffff', 'ce7e45', 'df923d', 'f1b555', 'fcd163', '99b718', '74a901',
    '66a000', '529400', '3e8601', '207401', '056201', '004c00', '023b01',
    '012e01', '011d01', '011301'
  ],
};

In [10]:
rgbvis= composite.map(lambda img: img.visualize(bands = ['NDVI_mean'],**Vis_para).clip(India_boundary))

In [15]:
gifParams = {
  'region': region,
  'dimensions': 600,
  'crs': 'EPSG:4326',
  'framesPerSecond': 10,
  'format':'gif',
  'addDate': 'yyyy-mm-dd'
}

In [12]:
print(rgbvis.getVideoThumbURL(gifParams))

https://earthengine.googleapis.com/v1/projects/shreenitya0428/videoThumbnails/7dcb79ca7ed9068c36f19d86ff40ec56-55c10cc726cad1fa13d46ea8bc8987fc:getPixels
